In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [2]:
load_dotenv()

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash")

In [3]:
class BlogState(TypedDict):

    title: str
    outline: str
    content: str

In [5]:
def create_outline(state: BlogState) -> BlogState:

    # fetch title
    title = state['title']

    # call llm gen outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    # update state
    state['outline'] = outline

    return state

In [4]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the follwing outline \n {outline}'

    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [6]:
graph = StateGraph(BlogState)

# nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

# edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

In [7]:
intial_state = {'title': 'Rise of AI in Bangladesh'}

final_state = workflow.invoke(intial_state)

print(final_state['outline'])

[{'type': 'text', 'text': 'Here is a detailed, structured blog outline on the topic **"The Rise of AI in Bangladesh: From Vision to Reality."**\n\n---\n\n# Blog Title Options\n1. **The Silicon Delta: How AI is Shaping the Future of Bangladesh**\n2. **From Digital to Smart: The Rise of Artificial Intelligence in Bangladesh**\n3. **Byte by Byte: How Bangladesh is Embracing the AI Revolution**\n\n---\n\n## **1. Catchy Introduction**\n* **The Hook:** Start with a striking stat or a vivid scenario (e.g., a farmer in rural Dinajpur using a smartphone app to detect crop diseases using AI vision).\n* **Context:** Briefly mention Bangladesh’s transition from the "Digital Bangladesh" vision to the ambitious "Smart Bangladesh 2041" roadmap.\n* **Thesis Statement:** AI is no longer just a buzzword in Bangladesh; it is becoming the driving engine behind economic growth, technological innovation, and societal transformation across key sectors.\n\n---\n\n## **2. The Foundations: Why is AI Booming in 